In [1]:
# Ensure required packages are installed in the notebook environment
import os
import shutil
import ssl
from doctr.models import ocr_predictor
from doctr.io import DocumentFile
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- CONFIGURATION ---
DB_DIR = "./chroma_db"
PDF_DATA_DIR = "./reference-health-files" # all PDFs needed for sourcing will be stored here
IMAGE_PATH = "john_doe_eob.jpg" # medical bill to analyze

In [3]:
# --- INITIALIZE MODELS ---
# Using all-MiniLM-L6-v2 for semantic vectors as per your notes
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# LLM
llm = Ollama(model="llama3.2", temperature=0, num_ctx=4096) # make sure to run ollama pull llama3.2

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10977.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/f4/gqrvf9kn21vfz20ybyccg4j80000gn/T/ipykernel_92767/3484588961.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.2", temperature=0, num_ctx=4096) # make sure to run ollama pull llama3.2


In [4]:
def build_knowledge_base():
    """Builds the vector database from PDFs in the reference folder."""
    if not os.path.exists(PDF_DATA_DIR):
        os.makedirs(PDF_DATA_DIR)
        print(f"Created {PDF_DATA_DIR}. Add your PDFs there and run again.")
        return None

    print(f"Loading PDFs from {PDF_DATA_DIR}...")
    loader = DirectoryLoader(PDF_DATA_DIR, glob="./*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    
    # Recursive chunking to keep semantic meaning intact
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    
    print(f"Vectorizing {len(chunks)} chunks into ChromaDB...")
    # Delete old DB if it exists to prevent 'readonly' errors
    if os.path.exists(DB_DIR):
        shutil.rmtree(DB_DIR)
        
    vector_db = Chroma.from_documents(
        documents=chunks, 
        embedding=embeddings, 
        persist_directory=DB_DIR
    )
    print("Knowledge base built successfully.")
    return vector_db

def extract_text_from_image(image_path):
    """Uses DocTR to turn a JPG into structured text."""
    print(f"Performing OCR on {image_path}...")
    model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
    doc = DocumentFile.from_images([image_path])
    result = model(doc)
    
    full_text = ""
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                line_text = " ".join([word.value for word in line.words])
                full_text += line_text + "\n"
    return full_text

In [5]:
def analyze_bill_with_rag(bill_text):
    """Uses RAG to explain the OCR text using the PDF knowledge base."""
    vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    
    template = """
    ### SYSTEM INSTRUCTION ###
    You are a Medical Billing Specialist. Use the provided context to analyze the bill.
    Be factual. If the info isn't in the context, say you aren't sure.

    ### RESEARCH CONTEXT ###
    {context}

    ### PATIENT BILL ###
    {question}

    ### ANALYSIS REPORT ###
    """
    PROMPT = PromptTemplate(template=template, input_variables=["context", "question"])
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
        chain_type_kwargs={"prompt": PROMPT}
    )
    
    print("Generating analysis...")
    response = qa_chain.invoke(bill_text)
    return response["result"]

In [6]:
# EXECUTION

if __name__ == "__main__":
    # Force fix for SSL errors on Mac if needed
    ssl._create_default_https_context = ssl._create_unverified_context

    # Step A: Ensure Knowledge Base is ready
    if not os.path.exists(DB_DIR):
        build_knowledge_base()
    
    # Step B: Check if image exists
    if os.path.exists(IMAGE_PATH):
        # OCR the image
        raw_text = extract_text_from_image(IMAGE_PATH)
        
        # Analyze the text
        analysis = analyze_bill_with_rag(raw_text)
        
        print("\n" + "="*40)
        print("MEDICAL BILL AUDIT REPORT")
        print("="*40)
        print(analysis)
    else:
        print(f"Error: {IMAGE_PATH} not found. Please place your bill JPG in the folder.")

Loading PDFs from ./reference-health-files...
Vectorizing 70 chunks into ChromaDB...
Knowledge base built successfully.
Performing OCR on john_doe_eob.jpg...


/var/folders/f4/gqrvf9kn21vfz20ybyccg4j80000gn/T/ipykernel_92767/696177441.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)


Generating analysis...

MEDICAL BILL AUDIT REPORT
Based on the provided context, I will analyze the bill as a Medical Billing Specialist.

The Explanation of Benefits (EOB) statement indicates that this is not a bill, but rather an EOB from Aetna Health Insurance. The EOB provides details about the claim and the patient's responsibility for the charges.

From the EOB, we can see that:

* The claim number is CLM-9928374-X.
* The issue date is 04/10/2026.
* The total amount due from the patient is $400.00, which includes a deductible and coinsurance.
* The remark code indicates that the charges exceed the fee schedule.

The EOB also lists two services:

1. EMERGENCY DEPT' VISIT (99284) - $800.00 allowed, $640.00 paid by plan
2. CTI HEAD W/O CONTRAST (70450) - $1,200.00 allowed, $960.00 paid by plan

The patient's responsibility for the charges is calculated as follows:

* Total amount due: $400.00
* Deductible and coinsurance: $400.00
* Patient pays: $0.00 (since the EOB indicates that t

In [18]:
# Initialize your local BioMistral
llm = Ollama(model="cniongolo/biomistral")

# Create the Retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 3}) # Get top 3 most relevant chunks

# The final chain: Input -> Retriever -> Prompt + Context -> LLM -> Answer
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

# Example usage:
query = "What does CPT code 99284 represent on this bill?"
response = qa_chain.invoke(query)
print(response["result"])

/var/folders/f4/gqrvf9kn21vfz20ybyccg4j80000gn/T/ipykernel_86201/1148767726.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="cniongolo/biomistral")


This is the billing code that refers to a preventive care service for adults age 51 and older. This 
code may be used in conjunction with the CPT code for an EKG, which can also be billed as a 
